# Snapshot の一覧と正確な件数比較

既定の Run All は、指定テーブルの保持 snapshot 一覧を1回取得します。比較は下の2つの ID を明示して `run_compare = True` にした場合だけ実行します。LLM は使用しません。

一覧の記録時刻は snapshot の時刻、取得完了時刻はこの読取の時刻です。保存結果は取得時点の証跡です。件数差は右−左であり、追加・削除された行の内訳や行内容の一致を示しません。


In [ ]:
from IPython.display import HTML, display
from tools.local_experiment.notebook_lab.query import (
    snapshots, compare_counts, recover_snapshots, render_snapshots,
)
from tools.local_experiment.notebook.client import NotebookError


In [ ]:
table = "smoke.fixture_v1"
listing_result = None
comparison_result = None
saved_result = None
try:
    listing_result = snapshots(table)
    display(HTML(render_snapshots(listing_result)))
except NotebookError as error:
    print("snapshot 一覧は取得できませんでした:", str(error))


一覧にある互換な snapshot ID を文字列で指定します。比較は左、右の順に COUNT を行い、両方の成功と同じテーブル UUID・schema を確認できた場合だけ差を表示します。失敗や中断で古い結果を採用しません。


In [ ]:
left_id = ""
right_id = ""
run_compare = False
comparison_result = None
saved_result = None
if run_compare:
    listing = listing_result.get("snapshots") if listing_result else None
    retained = {item["snapshot_id"]: item for item in listing["snapshots"]} if listing else {}
    if (not listing or listing["table"] != "iceberg." + table
            or not left_id or not right_id or left_id == right_id
            or any(selected not in retained or retained[selected]["schema_compatibility"] != "compatible"
                   for selected in (left_id, right_id))):
        print("一覧取得後、同じテーブルの互換な2つの snapshot ID を明示してください。")
    else:
        try:
            comparison_result = compare_counts(table, left_id, right_id)
            display(HTML(render_snapshots(comparison_result)))
        except NotebookError as error:
            print("件数比較は完了しませんでした。差は採用していません:", str(error))


## 保存結果の再表示

一覧または比較の実行 token を指定します。保存した証跡だけを検証して表示し、追加の一覧取得・COUNT・モデル呼出は行いません。


In [ ]:
snapshot_attempt = ""
saved_result = None
if snapshot_attempt:
    try:
        saved_result = recover_snapshots(snapshot_attempt)
        display(HTML(render_snapshots(saved_result)))
    except NotebookError as error:
        print("保存結果は確認できませんでした:", str(error))
